In [60]:
import base64
import io
import json
import os
from hashlib import md5

import open_clip
import torch
from PIL import Image
from torch.utils.data.dataloader import DataLoader

In [61]:
class ClipModel(torch.nn.Module):
    def __init__(self, model_name="ViT-B-32", pretrained="laion2b_s34b_b79k"):
        super().__init__()
        self.clip_backbone, _, self.preprocess = open_clip.create_model_and_transforms(model_name=model_name,
                                                                                       pretrained=pretrained)
        for p in self.clip_backbone.parameters():
            p.requires_grad = False
        self.clip_backbone.eval()
        self.tokenizer = open_clip.get_tokenizer('ViT-B-32')

        self.class_labels = ["class1", "class2", "class3", "class4"]
        self.class_tokens = self.tokenizer(self.class_labels)
        self.class_features = self.clip_backbone.encode_text(self.class_tokens)
        if torch.cuda.is_available():
            self.class_features /= self.class_features.norm(dim=-1, keepdim=True)
            self.class_features = self.class_features.cuda()

        self.head = torch.nn.Linear(self.clip_backbone.visual.output_dim,
                                    self.clip_backbone.visual.output_dim)
        if torch.cuda.is_available():
            self.clip_backbone.cuda()
            self.head.cuda()

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            image_features = self.clip_backbone.encode_image(images)
        image_features = self.head(image_features)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        class_similarity = (100.0 * image_features @ self.class_features.T).softmax(dim=-1)
        return class_similarity



In [62]:
class ApiResponsesDataset(torch.utils.data.Dataset):
    def __init__(self, clip_preprocess, img_dir="../img", responses_cache="../cache/responses.json"):
        super().__init__()
        self.clip_preprocess = clip_preprocess
        self.img_dir = img_dir
        self.responses_cache = responses_cache

        self.responses_json = json.load(open(responses_cache, "r"))
        self.raw_images = [Image.open(f"{img_dir}/{p}") for p in os.listdir(img_dir)]

        self.data = []
        self.labels = []

        self.preprocess()

    def preprocess(self):
        self.data = [self.clip_preprocess(raw_img).unsqueeze(0) for raw_img in self.raw_images]

        try:
            for raw_img in self.raw_images:
                buf = io.BytesIO()
                raw_img.save(buf, format="PNG")
                img_base64_bytes = base64.b64encode(buf.getvalue())
                img_hash = md5(img_base64_bytes).hexdigest()
                response = self.responses_json[img_hash]
                label = torch.Tensor([value for value in response['classes'].values()])
                self.labels.append(label)
        except KeyError:
            exit(f"Dataset integrity error. Not all images have responses.")

    def __len__(self):
        return len(self.raw_images)

    def __getitem__(self, index):
        return self.data[index], self.labels[index]

In [63]:
WEIGHTS_DIR = "../weights"
RANDOM_SEED = 161
BATCH_SIZE = 1

torch.random.manual_seed(RANDOM_SEED)  # set fixed seed to ensure determinism requirement
model = ClipModel()
dataset = ApiResponsesDataset(model.preprocess)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
